# Modelagem Estatística: Regressão Linear Múltipla (OLS)

Após a validação descritiva das hipóteses, o objetivo deste notebook é construir um modelo estatístico multivariável para quantificar o impacto simultâneo dos ofensores no atraso das entregas. 

A equação da Regressão Linear Múltipla assume a forma:
$$Y = \beta_0 + \beta_1X_1 + \beta_2X_2 + \dots + \beta_nX_n + \epsilon$$

Onde buscamos prever o **Atraso em Minutos** ($Y$) com base nas seguintes variáveis independentes ($X$):
1. **Distância (km):** O peso de cada quilômetro adicional.
2. **Turno:** O impacto de operar à Tarde ou à Noite (em comparação com a Manhã).
3. **Eventos (Dinâmica):** O efeito de ter ou não repasse financeiro ativado.

In [4]:
import pandas as pd
import statsmodels.api as sm
from pathlib import Path

In [10]:
# 1. Carregando Dados
processed_path = Path("../data/processed/base_tratada.csv")
df = pd.read_csv(processed_path)

# Selecionando apenas as colunas do modelo
colunas_modelo = ['atraso_min', 'km', 'turno', 'evento_flag']
df_modelo = df[colunas_modelo].dropna().copy()

print(f"Base de Modelagem preparada com {len(df_modelo)} registros.")

Base de Modelagem preparada com 13315 registros.


In [11]:
# --- PRÉ-PROCESSAMENTO: ONE-HOT ENCODING ---

# Transformando a variável categórica 'turno' em Dummies.
# Usamos drop_first=True para evitar a "Armadilha da Variável Dummy" (Multicolinearidade).
# O turno da 'Manhã' será a nossa categoria de referência (Baseline).
df_ols = pd.get_dummies(df_modelo, columns=['turno'], drop_first=True, dtype=int)

print("Variáveis disponíveis para o modelo:")
print(df_ols.columns.tolist())

Variáveis disponíveis para o modelo:
['atraso_min', 'km', 'evento_flag', 'turno_Noite', 'turno_Tarde']


In [12]:
# --- TREINAMENTO DO MODELO OLS ---

# 1. Definindo a variável dependente (Y) e as independentes (X)
y = df_ols['atraso_min']
X = df_ols[['km', 'evento_flag', 'turno_Tarde', 'turno_Noite']]

# 2. Adicionando a constante (o Beta 0 / Intercepto da equação)
X = sm.add_constant(X)

# 3. Ajustando o modelo de Regressão Linear por Mínimos Quadrados Ordinários
modelo_ols = sm.OLS(y, X).fit()

# 4. Imprimindo o sumário estatístico
print(modelo_ols.summary())

                            OLS Regression Results                            
Dep. Variable:             atraso_min   R-squared:                       0.151
Model:                            OLS   Adj. R-squared:                  0.150
Method:                 Least Squares   F-statistic:                     590.4
Date:                Sun, 15 Mar 2026   Prob (F-statistic):               0.00
Time:                        13:47:24   Log-Likelihood:                -59835.
No. Observations:               13315   AIC:                         1.197e+05
Df Residuals:                   13310   BIC:                         1.197e+05
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
                  coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------
const         -49.1918      0.407   -120.762      

### Resultados do Modelo OLS

* **Distância (O Maior Ofensor):** O coeficiente do `km` (4.2581) prova estatisticamente a Hipótese 1. **Cada 1 km adicional na rota adiciona, em média, 4,26 minutos ao tempo de entrega**. Em uma modalidade Expressa (SLA de 40 min), uma rota de 10 km consome matematicamente quase 43 minutos extras apenas pelo fator distância, garantindo o rompimento do SLA.
* **Dinâmica (`evento_flag`):** O modelo revelou um coeficiente positivo (+2.00) para os eventos. Isso não significa que o repasse financeiro "causa" atrasos, mas evidencia o comportamento reativo da plataforma: a dinâmica só é acionada em momentos atípicos de operação. Mesmo com o incentivo financeiro ativado, a operação ainda sofre uma penalidade residual de 2 minutos em relação a um cenário de normalidade.
* **Turno Noturno (`turno_Noite`):** Operar durante a noite adiciona, isoladamente, 1,57 minutos de atraso à entrega em comparação com o turno da manhã. Isso corrobora os achados da Hipótese 7, demonstrando que a escassez de frota e a alta demanda noturna afetam tanto a margem financeira quanto a performance temporal.
* **$R^2$:** O modelo explica 15,1% da variância dos atrasos ($R^2$ = 0.151). Embora pareça um valor conservador, é importante ressaltar que estamos lidando com dados que foram simulados sem compromisso de espelhar a realidade.